# Chapter 06: Events and Triggers in Depth (notebook edition)

## Learning Objectives

- Predict which workflows fire
- Explain tag, paths and token surprises
- Evaluate cron expressions

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Load the real experiments

Ten experiments were run on this repo; each records the workflows that really fired. The model predicts the same sets from the workflow files' `on:` blocks.

In [ ]:
import json
from intro_gha import WORKFLOWS_DIR
from intro_gha.events import would_fire, next_cron_run
from intro_gha.workflow import load_workflow
OBS = json.loads((ROOT/"fixtures"/"events_ch06_observed.json").read_text())
WFS = {n: load_workflow(WORKFLOWS_DIR / n) for n in OBS["workflows"]}
print(len(OBS["observations"]), "experiments")

## 2. Predict vs observed

For each experiment compare the predicted set with the real one. You should see no mismatch.

In [ ]:
for o in OBS["observations"]:
    present = o.get("workflows", OBS["workflows"])
    pred = sorted(n for n in present if would_fire(WFS[n], o["event"]))
    print(f'{o["id"]:28} {pred}')
    assert pred == sorted(o["fired"])

## 3. The three surprises, one at a time

A `paths` workflow firing on a tag; `branches`-only ignoring a tag; a token-created tag starting nothing.

In [ ]:
tag = {"name": "push", "ref": "refs/tags/demo-v1", "changed": []}
print("ch03 (paths only) on a tag:", would_fire(WFS["ch03-tally-ci.yml"], tag))
print("ci (branches only) on a tag:", would_fire(WFS["ci.yml"], tag))
print("same tag by GITHUB_TOKEN:", would_fire(WFS["ch06-events.yml"], {**tag, "by_github_token": True}))
assert would_fire(WFS["ch03-tally-ci.yml"], tag) and not would_fire(WFS["ci.yml"], tag)
assert not would_fire(WFS["ch06-events.yml"], {**tag, "by_github_token": True})

## 4. The pull request and workflow_run values

`github.sha` equals the merge commit; `workflow_run` runs on the default branch's commit.

In [ ]:
d = OBS["event_dumps"]
pr, wr = d["pull_request"], d["workflow_run"]
print(pr["sha"][:7], pr["pr_head_sha"][:7], wr["downstream_sha"][:7], wr["upstream_head_sha"][:7])
assert pr["sha"] == pr["api_merge_commit_sha"] != pr["pr_head_sha"]
assert wr["downstream_sha"] == wr["default_branch_head"] != wr["upstream_head_sha"]

## 5. Cron by hand and by code

Evaluating from 2026-10-05 12:00 UTC (a Monday).

In [ ]:
from datetime import datetime
start = datetime(2026, 10, 5, 12, 0)
for expr in ["23 4 1 1 *", "30 6 * * 1-5", "0 0 13 * 5"]:
    print(expr, "->", next_cron_run(expr, start))
assert next_cron_run("0 0 13 * 5", start) == datetime(2026, 10, 9, 0, 0)  # 13th OR a Friday

## Takeaways & Next Steps

- Silent non-triggers are the norm; model your filters before pushing.
- `github.sha` is the merge commit on PRs; `workflow_run` runs on the default branch.
- Next: Chapter 07, contexts and expressions.

## Chapter Link

Read: `learning_modules/chapter_06_events_and_triggers.md`